# 03 - Model Registration & Tracking

**Azure ML MLOps Workshop | Block 2 (continued)**

### Why a Model Registry?

Experiments produce many models. The Model Registry is where you promote the *best* ones
into a governed, versioned catalog:
- Each model version has full provenance (data version, experiment run, metrics)
- Models can be tagged by region, stage (staging/production), or use case
- Deployment always references a registered model - never a loose file

### What you'll do:
1. Identify the best run from experiments
2. Register the model in Azure ML Model Registry
3. Add metadata and tags
4. Query and manage model versions

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow
from mlflow.tracking import MlflowClient

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
mlflow_client = MlflowClient()
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Find the Best Experiment Run

Query the experiment runs from notebook 02 and select the best by F1 score.

In [ ]:
EXPERIMENT_NAME = "contoso-lead-classifier"
experiment = mlflow_client.get_experiment_by_name(EXPERIMENT_NAME)

all_runs = mlflow_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=20,
)

finished_runs = [r for r in all_runs if r.info.status == "FINISHED" and "f1" in r.data.metrics]
finished_runs.sort(key=lambda r: r.data.metrics.get("f1", 0), reverse=True)
best_run = finished_runs[0]

print(f"Best run:")
print(f"  Run ID:     {best_run.info.run_id}")
print(f"  Model type: {best_run.data.params.get('model_type', 'N/A')}")
print(f"  F1 score:   {best_run.data.metrics['f1']:.4f}")
print(f"  Accuracy:   {best_run.data.metrics['accuracy']:.4f}")
print(f"  Precision:  {best_run.data.metrics['precision']:.4f}")
print(f"  Recall:     {best_run.data.metrics['recall']:.4f}")

## Step 2: Register the Model

We'll register the best model from MLflow into the Azure ML Model Registry.
This creates a governed, versioned model artifact.

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

MODEL_NAME = "contoso-lead-classifier"

run_model_uri = f"runs:/{best_run.info.run_id}/model"

registered_model = ml_client.models.create_or_update(
    Model(
        name=MODEL_NAME,
        path=run_model_uri,
        type=AssetTypes.MLFLOW_MODEL,
        description=(
            f"Contoso inspection lead classifier. "
            f"Trained on classified-inspections v2. "
            f"Model type: {best_run.data.params.get('model_type', 'N/A')}. "
            f"F1: {best_run.data.metrics['f1']:.4f}"
        ),
        tags={
            "task": "text-classification",
            "language": "en",
            "region": "region-1",
            "data_asset": "classified-inspections",
            "data_version": "2",
            "source_run_id": best_run.info.run_id,
            "f1_score": f"{best_run.data.metrics['f1']:.4f}",
            "stage": "staging",
        },
    )
)

print(f"Registered model: {registered_model.name}")
print(f"  Version: {registered_model.version}")
print(f"  Tags: {registered_model.tags}")

## Step 3: Query the Model Registry

Demonstrate how teams can discover and inspect registered models.

In [ ]:
print(f"All versions of '{MODEL_NAME}':")
for model in ml_client.models.list(name=MODEL_NAME):
    print(f"  v{model.version} | F1: {model.tags.get('f1_score', 'N/A')} | "
          f"Stage: {model.tags.get('stage', 'N/A')} | "
          f"Region: {model.tags.get('region', 'N/A')}")

In [ ]:
model_v1 = ml_client.models.get(name=MODEL_NAME, version="1")

print(f"Model: {model_v1.name} v{model_v1.version}")
print(f"  Description: {model_v1.description}")
print(f"  Path: {model_v1.path}")
print(f"\nFull lineage:")
print(f"  Data asset: {model_v1.tags.get('data_asset')} v{model_v1.tags.get('data_version')}")
print(f"  Training run: {model_v1.tags.get('source_run_id')}")
print(f"  F1 at training: {model_v1.tags.get('f1_score')}")

## Step 4: Multi-Region Model Management

In a multi-region setup (across multiple regions), you have options:

**Option A - Shared Registry:** All regions register models to one workspace. Use tags to filter.

**Option B - Federated Registries:** Each region has its own workspace. Use Azure ML Registry (preview) for cross-workspace sharing.

Let's simulate how tagging enables Option A:

In [ ]:
print("Models for Region 1:")
for model in ml_client.models.list(name=MODEL_NAME):
    if model.tags.get("region") == "region-1":
        print(f"  {model.name} v{model.version} | F1: {model.tags.get('f1_score')}")

print("\nModels in staging:")
for model in ml_client.models.list(name=MODEL_NAME):
    if model.tags.get("stage") == "staging":
        print(f"  {model.name} v{model.version} | Region: {model.tags.get('region')}")

## Go to Azure ML Studio Now

Navigate to **Models** in the left navigation and explore:

1. Click on **contoso-lead-classifier** to see all versions
2. Click on a version to see its details, tags, and linked artifacts
3. Click **Job** link to trace back to the experiment run
4. From the run, click **Data** to trace back to the data asset version

This is the full lineage chain: **Data v2 -> Experiment Run -> Model v1**

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Model Registry** | Promoted best experiment model to governed catalog |
| **Versioning** | Each registration creates an immutable version |
| **Lineage** | Tags link model -> run -> data version |
| **Multi-region** | Tags enable filtering by region/stage |
| **Governance** | Only registered models can be deployed |

---

**Next:** [04 - Model Deployment](./04_model_deployment.ipynb)